# Amazon rivers-on vertical structure and plume-direction transect

This notebook derives the mean offshore plume direction from the day-80 surface dye centroid, constructs a source-to-plume transect, plots vertical dye/salinity sections, and computes cumulative dye inventory above each model depth.

In [ ]:
using Oceananigans
using CairoMakie
using Printf

data_directory = raw"C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\repo-cleanup\ocean-modeling\amazon_river\new jld2 files\no spinups\80 day salinity dye"
dye_file = joinpath(data_directory, "amazon_rivers_on_80d_dye.jld2")
salinity_file = joinpath(data_directory, "amazon_rivers_on_80d_salinity_3d.jld2")

for file in (dye_file, salinity_file)
    @assert isfile(file) "Missing input file: $file"
end

dye = FieldTimeSeries(dye_file, "dye"; backend=OnDisk())
salinity = FieldTimeSeries(salinity_file, "S"; backend=OnDisk())
@assert length(dye.times) == length(salinity.times)
days_saved = Float64.(dye.times) ./ 86400

source_longitude = -49.5
source_latitude = 0.16
println("Loaded $(length(days_saved)) outputs through day $(last(days_saved)).")

In [ ]:
# Coordinates, spherical volume weights, and active-ocean mask.
longitude, latitude, depth = nodes(dye.grid, Center(), Center(), Center())
longitude_faces, latitude_faces, depth_faces = nodes(dye.grid, Face(), Face(), Face())
earth_radius = 6.371e6

delta_longitude = diff(deg2rad.(longitude_faces))
delta_sin_latitude = diff(sin.(deg2rad.(latitude_faces)))
delta_depth = diff(depth_faces)

cell_area = earth_radius^2 .*
            reshape(delta_longitude, length(delta_longitude), 1) .*
            reshape(delta_sin_latitude, 1, length(delta_sin_latitude))

cell_volume = cell_area .* reshape(delta_depth, 1, 1, length(delta_depth))

initial_salinity = Float64.(Array(interior(salinity[1])))
wet_cell = isfinite.(initial_salinity) .& (initial_salinity .> 0)
surface_wet = wet_cell[:, :, end]
nothing

## Derive the plume direction and construct the transect

The direction is the straight line from the release point toward the day-80 surface dye centroid. It is a reproducible bulk transport direction, not a claim that every part of the plume follows one path.

In [ ]:
direction_day = 80.0
direction_index = argmin(abs.(days_saved .- direction_day))
direction_dye = Float64.(Array(interior(dye[direction_index])))[:, :, end]
direction_dye[.!surface_wet] .= 0
surface_weight = direction_dye .* cell_area

centroid_longitude = sum(surface_weight .* reshape(longitude, length(longitude), 1)) / sum(surface_weight)
centroid_latitude = sum(surface_weight .* reshape(latitude, 1, length(latitude))) / sum(surface_weight)

# Convert centroid displacement to local east/north kilometers.
eastward_km = (centroid_longitude - source_longitude) * 111.32 * cosd(source_latitude)
northward_km = (centroid_latitude - source_latitude) * 111.32
direction_norm = hypot(eastward_km, northward_km)
eastward_unit = eastward_km / direction_norm
northward_unit = northward_km / direction_norm
direction_degrees = mod(rad2deg(atan(eastward_km, northward_km)), 360)

# Extend from the mouth in the centroid direction, stopping at the model edge.
candidate_distance_km = collect(0.0:5.0:1000.0)
candidate_longitude = source_longitude .+ candidate_distance_km .* eastward_unit ./ (111.32 * cosd(source_latitude))
candidate_latitude = source_latitude .+ candidate_distance_km .* northward_unit ./ 111.32
inside_domain = (candidate_longitude .>= minimum(longitude)) .&
                (candidate_longitude .<= maximum(longitude)) .&
                (candidate_latitude .>= minimum(latitude)) .&
                (candidate_latitude .<= maximum(latitude))

transect_distance_km = candidate_distance_km[inside_domain]
transect_longitude = candidate_longitude[inside_domain]
transect_latitude = candidate_latitude[inside_domain]

# Use the nearest horizontal model cell at each transect point.
transect_i = [argmin(abs.(longitude .- value)) for value in transect_longitude]
transect_j = [argmin(abs.(latitude .- value)) for value in transect_latitude]

@printf("Day-80 surface centroid: %.3f deg E, %.3f deg N\n", centroid_longitude, centroid_latitude)
@printf("Centroid displacement: %.1f km\n", direction_norm)
@printf("Transect bearing: %.1f deg clockwise from north\n", direction_degrees)
@printf("Transect length inside domain: %.1f km\n", last(transect_distance_km))

In [ ]:
# Verify the data-derived transect on the day-80 surface plume.
map_dye = copy(direction_dye)
map_dye[.!surface_wet] .= NaN

map_figure = Figure(size=(900, 650))
map_axis = Axis(map_figure[1, 1], xlabel="Longitude", ylabel="Latitude", title="Day-80 plume and data-derived transect")
map_plot = heatmap!(map_axis, longitude, latitude, log10.(max.(map_dye, 1e-12)); colorrange=(-6, 0), colormap=:viridis)
lines!(map_axis, transect_longitude, transect_latitude, color=:white, linewidth=4, label="Transect")
scatter!(map_axis, [source_longitude], [source_latitude], color=:red, marker=:star5, markersize=20, label="Release")
scatter!(map_axis, [centroid_longitude], [centroid_latitude], color=:cyan, marker=:diamond, markersize=16, label="Day-80 centroid")
axislegend(map_axis, position=:rt)
Colorbar(map_figure[1, 2], map_plot, label="log10 surface dye")
map_figure

## Upper-300-m transects

Colors show log10 dye concentration. White contours show 30, 32, and 34 PSU. Land or cells below the seafloor are blank.

In [ ]:
transect_days = [10.0, 30.0, 80.0]
transect_indices = [argmin(abs.(days_saved .- day)) for day in transect_days]
vertical_figure = Figure(size=(1500, 500))

for (panel, time_index) in enumerate(transect_indices)
    dye_field = Float64.(Array(interior(dye[time_index])))
    salinity_field = Float64.(Array(interior(salinity[time_index])))

    dye_section = fill(NaN, length(transect_distance_km), length(depth))
    salinity_section = fill(NaN, length(transect_distance_km), length(depth))

    for point in eachindex(transect_distance_km), k in eachindex(depth)
        i = transect_i[point]
        j = transect_j[point]
        if wet_cell[i, j, k]
            dye_section[point, k] = dye_field[i, j, k]
            salinity_section[point, k] = salinity_field[i, j, k]
        end
    end

    log_dye_section = log10.(max.(dye_section, 1e-12))
    axis = Axis(
        vertical_figure[1, panel],
        xlabel="Distance from Amazon mouth (km)",
        ylabel=panel == 1 ? "Depth (m)" : "",
        title="Day $(round(days_saved[time_index], digits=1))"
    )
    heatmap!(axis, transect_distance_km, depth, log_dye_section; colorrange=(-6, 0), colormap=:viridis)
    contour!(axis, transect_distance_km, depth, salinity_section; levels=[30, 32, 34], color=:white, linewidth=2)
    hlines!(axis, [-100], color=:red, linestyle=:dash, linewidth=2)
    ylims!(axis, -300, 0)
end

Colorbar(vertical_figure[1, 4], limits=(-6, 0), colormap=:viridis, label="log10 dye")
vertical_figure

## Cumulative dye inventory above depth

Each curve reports the fraction of dye located above each model-cell depth. A vertical reference line at 100 m highlights the fixed upper-ocean metric.

In [ ]:
inventory_days = [10.0, 30.0, 60.0, 80.0]
inventory_indices = [argmin(abs.(days_saved .- day)) for day in inventory_days]
cumulative_fraction_above = zeros(length(depth), length(inventory_indices))

for (column, time_index) in enumerate(inventory_indices)
    dye_field = Float64.(Array(interior(dye[time_index])))
    dye_field[.!isfinite.(dye_field)] .= 0
    dye_field[.!wet_cell] .= 0

    layer_inventory = [sum(dye_field[:, :, k] .* cell_volume[:, :, k]) for k in eachindex(depth)]
    total_inventory = sum(layer_inventory)

    for k in eachindex(depth)
        cumulative_fraction_above[k, column] = sum(layer_inventory[k:end]) / total_inventory
    end
end

inventory_figure = Figure(size=(850, 700))
inventory_axis = Axis(
    inventory_figure[1, 1],
    xlabel="Fraction of total dye above depth",
    ylabel="Depth (m)",
    title="Cumulative vertical dye inventory"
)

for (column, day) in enumerate(inventory_days)
    scatterlines!(inventory_axis, cumulative_fraction_above[:, column], depth, linewidth=3, markersize=7, label="Day $(Int(day))")
end

hlines!(inventory_axis, [-100], color=:red, linestyle=:dash, linewidth=2, label="100 m")
xlims!(inventory_axis, 0, 1.02)
ylims!(inventory_axis, -500, 0)
axislegend(inventory_axis, position=:lb)
inventory_figure

In [ ]:
# Print fixed-depth retention values for a compact results table.
report_depths = [-30.0, -50.0, -100.0, -200.0, -500.0]

@printf("%-8s", "Day")
for boundary in report_depths
    @printf(" %12s", "above $(Int(abs(boundary)))m")
end
println()

for (column, day) in enumerate(inventory_days)
    @printf("%-8.0f", day)
    for boundary in report_depths
        included_levels = findall(depth .>= boundary)
        deepest_included_level = first(included_levels)
        @printf(" %12.5f", cumulative_fraction_above[deepest_included_level, column])
    end
    println()
end